<a href="https://colab.research.google.com/github/SiriusDarkz/riesgo-mora-cooperativa/blob/main/caso1_cooperativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso 1 · Riesgo de mora en préstamos nuevos
## Cooperativa Progreso del Sur

**Asignatura:** Selección y Validación de Modelos  
**Profesor:** Dr. Edian Franco  
**Equipo:** Jose Eugenio Duran Vizcaino · Anthony Burgos · Isaac Sanchez ·
Maximo Martinez · Francisco Jose Mejia

---

## El caso

La **Cooperativa Progreso del Sur** ofrece préstamos personales, comerciales
y para mejoras de vivienda a través de sucursales en Santo Domingo, San
Cristóbal, Baní y Azua. Durante el último año aumentaron las solicitudes
recibidas por canales digitales, pero el equipo de riesgo continúa revisando
manualmente buena parte de los expedientes.

La gerencia observa que algunas personas presentan atrasos importantes durante
los primeros meses del préstamo. Cuando esto ocurre, la cooperativa debe
realizar llamadas de cobro, renegociar condiciones y aumentar las provisiones
financieras. La gerente de riesgo plantea la necesidad de **identificar cuáles
solicitudes nuevas podrían presentar una mora superior a 30 días durante sus
primeros seis meses**.

Dos restricciones definen el problema:

- **Capacidad limitada:** el equipo de analistas solo puede revisar en
  detalle 120 solicitudes por semana.
- **Sin rechazo automático:** la gerencia no desea rechazar solicitantes con
  el modelo; desea priorizar cuáles expedientes necesitan verificación
  adicional.

## Qué construye este proyecto

> Un procedimiento que, cada semana, ordena las solicitudes nuevas por riesgo
> de mora y selecciona las 120 que el equipo de analistas debe revisar en
> detalle. La revisión funciona como tratamiento preventivo: verifica ingresos,
> pide garantías o ajusta condiciones, y así evita una parte de las moras que
> iban a ocurrir. El modelo no aprueba ni rechaza a nadie, solo apunta la
> capacidad limitada de revisión hacia donde más pérdida puede prevenir. Se
> recomendará implementarlo únicamente si demuestra, en un test honesto, que
> apunta mejor que la regla actual de la cooperativa.

Todo el ejercicio utiliza **datos sintéticos** generados en Python con semilla
fija, siguiendo las 8 etapas del protocolo del curso: formular, generar datos,
auditar variables, diseñar la evaluación, comparar contra baselines, congelar
el protocolo, evaluar en test una sola vez y recomendar.

# Etapa 1 · Formulación del problema

**Actor.** La gerencia de riesgo de la Cooperativa Progreso del Sur: la
gerente de riesgo define la política de revisión y su equipo de analistas la
ejecuta.

**Decisión.** Cuáles solicitudes nuevas de préstamo se envían cada semana a
verificación adicional, dentro del límite operativo de 120 revisiones
semanales.

**Acción.** Revisión manual detallada del expediente, que puede derivar en
verificación de ingresos, solicitud de garantías, reducción del monto o
cambio del plazo. La predicción no rechaza solicitantes: prioriza cuáles
revisar.

**Unidad de análisis.** Una solicitud de préstamo (una fila = una solicitud).
No es el cliente: un mismo cliente puede presentar varias solicitudes, lo que
obliga a controlar que sus solicitudes no queden repartidas entre desarrollo
y test.

**Momento de predicción.** Al recibir la solicitud completa, antes de la
decisión de aprobación. Elegimos este momento (y no "antes del desembolso")
porque la revisión adicional sirve precisamente para informar las condiciones
de aprobación. Consecuencias: (a) la variable `approved` aún no existe al
predecir, por lo que no puede usarse como predictora; (b) solo las
solicitudes aprobadas y desembolsadas llegan a tener etiqueta observada — una
limitación (etiquetas selectivas) que declaramos en el informe final.

**Horizonte.** Los primeros 6 meses de vida del préstamo, contados desde el
desembolso. La etiqueta de un préstamo solo se conoce cuando esta ventana se
cierra. *Supuesto de madurez:* asumimos que el análisis se realiza en una
fecha en la que todos los préstamos simulados ya completaron su ventana de
6 meses; en producción, la cooperativa solo podría entrenar con solicitudes
desembolsadas al menos 6 meses atrás, y las más recientes aún no tendrían
etiqueta observada.

**Variable objetivo.** `default_30d`: vale 1 si el préstamo alcanza una mora
superior a 30 días en algún momento durante sus primeros 6 meses; 0 en caso
contrario. Se deriva del seguimiento de atrasos (`days_past_due_6m`). Nótese
que combina dos números con roles distintos: los 30 días definen la severidad
del atraso que cuenta como evento; los 6 meses definen la ventana de
observación.

**Capacidad operativa.** 120 solicitudes por semana pueden recibir revisión
detallada. Esto convierte el problema en uno de **priorización**: el
procedimiento ordena las solicitudes de cada semana por riesgo estimado y las
120 primeras se revisan.

**Costo de los errores.**
- *Falso negativo* (no revisar una solicitud que luego cae en mora): costo
  financiero directo — provisiones, gestión de cobranza, renegociación y
  posible pérdida de capital.
- *Falso positivo* (gastar una revisión en una solicitud que habría pagado
  bien): horas de analista y fricción para un buen socio. Con capacidad fija,
  cada falso positivo tiene además un costo de oportunidad: desplaza del top
  120 a una solicitud riesgosa.

**Supuestos de costo (ilustrativos, fijados antes de la evaluación).**
Para traducir los errores a magnitudes comparables adoptamos supuestos
redondos, coherentes con los datos sintéticos que generaremos:

| Concepto | Supuesto |
|---|---|
| Monto promedio del préstamo | RD\$150,000 |
| Pérdida esperada si hay mora >30d (provisiones, cobranza, pérdida) | ≈20% del monto → RD\$30,000 |
| Costo de una revisión manual (≈2 horas de analista) | RD\$1,000 |
| Efecto de la revisión sobre una solicitud riesgosa | reduce la probabilidad de mora ≈35% |

Implicación: un falso negativo cuesta ~30 veces más que un falso positivo;
por eso consideramos más costoso el falso negativo y la métrica principal
medirá cuántas moras reales se capturan dentro de la capacidad semanal.
Además, el beneficio esperado de revisar un caso realmente riesgoso
(0.35 × RD\$30,000 ≈ RD\$10,500) supera con holgura el costo de la revisión
(RD\$1,000), lo que valida usar la capacidad completa.

El 35% es un **parámetro de diseño de la simulación**: el enunciado exige que
la revisión pueda disminuir la mora observada; nosotros fijamos su magnitud
en un valor moderado y plausible, y el generador de datos de la Etapa 2 usará
este mismo parámetro. Como el tratamiento es idéntico para cualquier método
de selección, la comparación entre el modelo y la regla vigente no depende
del valor exacto: variaciones razonables (20%–50%) cambian las cifras en
pesos, no la conclusión. Estas cifras son supuestos del ejercicio; en una
implementación real se calibrarían con la contabilidad de la cooperativa y
las normas de provisión aplicables.

**Criterio de no implementación.** El procedimiento no se recomienda si, con
las mismas 120 revisiones semanales en el período de test, no captura más
moras futuras que la regla operativa vigente (priorizar solicitudes con
deuda/ingreso alta y atrasos previos). Tampoco si la mejora, traducida a
dinero con los supuestos de costo anteriores, resulta demasiado pequeña para
compensar el costo de construir, mantener y monitorear el modelo.

# Etapa 3 · Auditoría de variables

Clasificamos cada variable del dataset según el papel que puede cumplir en el
proyecto. El criterio es el mismo para todas, la pregunta del enunciado:

> ¿Esta variable existiría, con el mismo valor y significado, al momento de
> decidir?

El momento de decidir quedó definido en la Etapa 1: cuando la solicitud llega
completa, antes de aprobarla. Hicimos esta auditoría antes de programar el
generador de datos, porque la clasificación de cada variable indica cómo debe
generarse: las de fuga se generan a partir del resultado, la intervención
modifica la probabilidad de mora, y la ambigua necesitaba una definición
antes de poder existir.

| # | Variable | Clasificación | Justificación |
|---|---|---|---|
| 1 | `application_date` | Identificador (eje temporal) | Existe al decidir, pero no describe al solicitante: sirve para ordenar las solicitudes en el tiempo, agruparlas por semana y hacer la partición temporal. No se usa como predictor directo. |
| 2 | `branch_id` | Predictora potencialmente válida | La sucursal se conoce desde que entra la solicitud y no cambia. También se usa para calcular el baseline histórico. Variable sensible por territorio (ver nota 3). |
| 3 | `client_id` | Identificador | Existe al decidir, pero solo identifica a la persona. No entra al modelo; sirve para que un mismo cliente no quede repartido entre desarrollo y test. |
| 4 | `age` | Predictora potencialmente válida | Viene en el formulario, así que está disponible al decidir. Variable sensible (ver nota 3). |
| 5 | `monthly_income` | Predictora potencialmente válida | Es el ingreso que la persona declara al solicitar; ese valor existe al decidir. El ingreso verificado por la revisión aparece después, así que no es este. |
| 6 | `requested_amount` | Predictora potencialmente válida | Es el monto que el cliente pide en la solicitud. El monto aprobado puede terminar siendo otro, pero ese llega después. |
| 7 | `loan_term_months` | Predictora potencialmente válida | Plazo que el cliente solicita, disponible al decidir. El plazo final puede cambiar tras la revisión. |
| 8 | `existing_debt` | Predictora potencialmente válida | Deuda que el cliente tiene al momento de solicitar, según buró o declaración. Existe al decidir. |
| 9 | `employment_type` | Predictora potencialmente válida | Se declara en el formulario, disponible al decidir. |
| 10 | `months_in_job` | Predictora potencialmente válida | Se declara en la solicitud, disponible al decidir. Viene vacía cuando el empleo es informal, porque no hay nómina que la acredite. |
| 11 | `prior_late_payments` | Predictora potencialmente válida | Cuenta atrasos de préstamos anteriores a esta solicitud. Es historial pasado, así que existe al decidir. |
| 12 | `payment_history_score` | Ambigua → válida bajo definición | Depende de cuándo se calcule. Si el sistema lo recalcula con el tiempo, el valor guardado hoy no es el que existía al decidir, y usarlo sería fuga. Lo definimos como el score calculado solo con historial previo y congelado en la fecha de la solicitud; con esa definición sí existe al decidir. Queda vacío para clientes nuevos. |
| 13 | `debt_to_income` | Predictora potencialmente válida | Se calcula con dos datos disponibles al decidir (deuda entre ingreso). Es además la base de la regla actual de la cooperativa. |
| 14 | `manual_review` | Intervención | No es un dato del solicitante: es la acción que la cooperativa toma sobre el expediente, y cambia el resultado porque la revisión reduce la probabilidad de mora. No entra al modelo; se analiza aparte. |
| 15 | `approved` | Posterior al momento de predicción | Como predecimos antes de aprobar, en ese momento esta variable todavía no existe. Solo sirve para saber qué solicitudes llegaron a tener etiqueta (las aprobadas y desembolsadas). |
| 16 | `days_past_due_6m` | Fuga como predictor · origen del target · solo auditoría | Es el resultado de los primeros 6 meses: no existe al decidir, y usarla para predecir sería fuga. Pero también tiene que estar en el dataset, porque de ella sale el target. Por eso el enunciado la lista dos veces: prohibida como predictor, obligatoria como origen de la etiqueta. |
| 17 | `default_30d` | Target | Es lo que queremos predecir. Se usa como etiqueta al entrenar y para evaluar; nunca como predictor. |
| 18 | `collection_calls_6m` | Fuga → excluir | Cobranza llama cuando ya hay atraso: el dato aparece después del resultado y por causa de él. Al decidir no existe. Se conserva solo para la demostración de fuga. |
| 19 | `restructured_after_default` | Fuga → excluir | Solo puede existir si el préstamo ya cayó en incumplimiento. Ocurre después del resultado. |
| 20 | `legal_collection_status` | Fuga → excluir | La cobranza legal empieza meses después de iniciada la mora. Al decidir no existe. |
| 21 | `p_mora_sin_intervencion` | Solo para auditoría (diseño propio) | Columna que agregamos nosotros: guarda la probabilidad de mora antes de aplicar el efecto de la revisión. En la vida real no existe; aquí sirve para medir cuánto cambia la intervención a las etiquetas. Nunca entra al modelo. |

**Resultado:** 11 variables candidatas a predictoras (10 válidas + la ambigua
ya definida), 2 identificadores, 1 intervención, 1 posterior (`approved`),
**las 4 variables de fuga del enunciado** (3 que se excluyen por completo y
`days_past_due_6m`, que además es el origen del target), 1 target y 1 columna
de auditoría propia.

**Notas**

1. `days_past_due_6m` aparece en las dos listas del enunciado (mínimas y de
   fuga). No hay contradicción, porque las listas responden preguntas
   distintas: la variable debe existir en el dataset, porque de ella sale el
   target, y a la vez no puede usarse como predictor, porque es el resultado
   mismo. Estar en los datos y entrar al modelo son cosas distintas.
2. `payment_history_score` no se podía clasificar sin antes definirla. La
   decisión del grupo quedó registrada en la tabla: score congelado a la
   fecha de la solicitud.
3. `age` y `branch_id` son válidas técnicamente pero sensibles (edad y
   territorio). En la Etapa 7 revisaremos los errores del modelo por
   sucursal, y el informe final discutirá las implicaciones de usarlas.
4. En la Etapa 5 entrenaremos un modelo con las variables de fuga incluidas,
   solo sobre la partición de desarrollo, para mostrar el desempeño casi
   perfecto y falso que producen, comparado con el modelo limpio. Los
   resultados quedarán en `reports/auditoria_fuga.md`.